## First stage: Claim Extractor

This first stage is already done by the LLM's pipeline, we will have to evaluate the outputs of the models Qwen3-1.7b (RAG) and Qwen3-4b (LoRA and RAG)
the output is the the folder Knowledge graph, in this case on the folder doc_146120936:
- LoRA-Output-4B_doc_146120936.json (Qwen3-4b)
- RAG-Output-4B_doc_146120936.json (Qwen3-4b)
- RAG-Output_doc_146120936.json (Qwen3-1.7b)

In [7]:
import glob

import json

import os
from huggingface_hub import InferenceClient
from groq import Groq

In [8]:
GROQ_KEY = os.getenv("GROQ_API_KEY")


In [23]:
FILES = glob.glob("./Knowledge-graph/doc_146120936/*-Output*.json")

Answers = []
num_questions = 0

for file in FILES:
    print("Processing file:", file)
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)
        for result in data["results"]:
            question = result["question"]

            num_questions += 1

            t = result["triplets"]

            # Case 1: {"triplets": [...], "raw_output": "..."}
            if isinstance(t, dict) and "triplets" in t:
                triplets = t["triplets"]

            # Case 2: ["[subject:..., ...]"]
            elif isinstance(t, list):
                triplets = t

            # Fallback
            else:
                triplets = []

            print("Question:", question)
            print("Triplets:", triplets)
    
print("Total questions processed:", num_questions)
        

Processing file: ./Knowledge-graph/doc_146120936\LoRA-Output-4B_doc_146120936.json
Question: What is a subtask of dense prediction tasks?
Triplets: []
Question: What is another name for Content-Aware ReAssembly of FEatures?
Triplets: ['[subject:Content-Aware ReAssembly of FEatures, Synonym-Of, CARAFE]']
Question: What method is used for Feature Pyramid Network?
Triplets: []
Question: What method is used for U-Net?
Triplets: []
Question: What method is used for instance segmentation?
Triplets: []
Question: What task is used for dense prediction tasks?
Triplets: ['[subject:Dense Prediction Task, Used-For, Task]']
Question: Which method is part of Faster RCNN?
Triplets: []
Question: Which method is compared with Global&Local?
Triplets: []
Question: Which dataset is used to evaluate Faster RCNN?
Triplets: ['[subject:Faster RCNN, Evaluated-With, MS COCO]']
Question: Which dataset is used to evaluate Global&Local?
Triplets: []
Question: Which dataset benchmarks image inpainting?
Triplets: []

In [ ]:
json_example = {
    "results": [
        {
            "question": "What is another name for Content-Aware ReAssembly of FEatures?",
            "triplets": {
                "triplets": [
                    "[subject:Content-Aware ReAssembly of FEatures, Synonym-Of, CARAFE]"
                ]
            }
        }
    ]
}

Answers = []
num_questions = 0


for result in json_example["results"]:
    question = result["question"]

    num_questions += 1

    t = result["triplets"]

    # Case 1: {"triplets": [...], "raw_output": "..."}
    if isinstance(t, dict) and "triplets" in t:
        triplets = t["triplets"]

    # Case 2: ["[subject:..., ...]"]
    elif isinstance(t, list):
        triplets = t

    # Fallback
    else:
        triplets = []

    print("Question:", question)
    print("Triplets:", triplets)
    

prompt = f"""
You are a helpful assistant that will verify the answer to a question based on provided knowledge triplets.

You will be provided with:
1. A question.
2. A set of knowledge triplets in the format: [subject:..., predicate:..., object:...].
3. A gold standard answer to the question.

You can answer in 3 ways:
- Supported — the triplet matches the information in the gold standard;
- Contradicted — the triplet conflicts with the gold standard;
- Not verifiable — the triplet cannot be confirmed using the available information.

<QUESTION>
{question}
</QUESTION>
<TRIPLETS>
{triplets}
</TRIPLETS>
<GOLD_STANDARD_ANSWER>
["CARAFE:Method","Synonym-Of","Content - Aware ReAssembly of FEatures:Method"],
</GOLD_STANDARD_ANSWER>

output only:
0 - supoported
1 - contradicted
2 - not verifiable
""" 

client = Groq(api_key=GROQ_KEY)
completion = client.chat.completions.create(
    model="groq/compound",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)
print(completion.choices[0].message.content)

Question: What is another name for Content-Aware ReAssembly of FEatures?
Triplets: ['[subject:Content-Aware ReAssembly of FEatures, Synonym-Of, CARAFE]']
0
